# Advanced: Multi-Strategy Attribute Suppression with PAMOLA.CORE

**Goal**: Master all 3 attribute suppression strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Risk-based suppression using k-anonymity thresholds
- **Strategy 2**: Conditional suppression based on field values
- **Strategy 3**: Multi-condition suppression with AND/OR logic
- Calculate privacy protection metrics
- Analyze data utility retention
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of privacy concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── anonymization/suppression/
        └── 02_attribute_suppression_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.anonymization.suppression.attribute_op import AttributeSuppressionOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 12 columns)
- Sample records (first 5 rows)
- Column statistics (types, ranges, unique counts)

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate realistic patient records with varying risk levels
    df = pd.DataFrame({
        'patient_id': [f'P{i:04d}' for i in range(1, n_records + 1)],
        'name': [f'Patient {i}' for i in range(1, n_records + 1)],
        'ssn': [f'{np.random.randint(100, 999)}-{np.random.randint(10, 99)}-{np.random.randint(1000, 9999)}' for _ in range(n_records)],
        'date_of_birth': pd.date_range('1940-01-01', periods=n_records, freq='30D'),
        'age': np.random.randint(18, 85, n_records),
        'phone_number': [f'555-{np.random.randint(100, 999)}-{np.random.randint(1000, 9999)}' for _ in range(n_records)],
        'email': [f'patient{i}@email.com' for i in range(1, n_records + 1)],
        'diagnosis': np.random.choice(['Diabetes', 'Hypertension', 'Asthma', 'Arthritis', 'Heart Disease'], n_records),
        'admission_date': pd.date_range('2024-01-01', periods=n_records, freq='8H'),
        'insurance_id': [f'INS{np.random.randint(100000, 999999)}' for _ in range(n_records)],
        'risk_level': np.random.choice(['Low', 'Medium', 'High'], n_records, p=[0.6, 0.3, 0.1]),
        'k_anonymity': np.random.randint(1, 20, n_records)  # Simulated k-anonymity score
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    elif pd.api.types.is_datetime64_any_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].min()} to {df[col].max()}")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names for each strategy
   - `"strategy1_field": "resume_id"` → YOUR primary column
   - `"strategy2_field": "phone"` → YOUR primary column
   - `"strategy3_field": "email"` → YOUR primary column
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Unique value counts per field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must exist in dataset before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "resume_id",            # Risk-based suppression
    "strategy2_field": "phone",   # Conditional suppression
    "strategy3_field": "email"           # Multi-condition suppression
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    print(f"  ✓ {strategy:20s}: '{field_name}' ({df[field_name].nunique()} unique values)")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="suppression_advanced_001",
    task_type="multi_strategy_suppression",
    description="Multi-strategy attribute suppression",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Risk-Based Suppression with K-Anonymity

**How to use:**
- Suppresses attributes for high-risk records (k < 5)
- Run to execute risk-based strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → identify risk → suppress → metrics → visualize → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `ka_risk_field='k_anonymity'`: Field containing k-anonymity scores
- `risk_threshold=5`: Suppress if k < 5 (vulnerable records)
- `additional_fields=['first_name']`: Also suppress related identifiers
- `save_suppressed_schema=True`: Save metadata about removed columns

**Note:** Only removes columns for vulnerable records, protects high-risk individuals

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: RISK-BASED SUPPRESSION WITH K-ANONYMITY")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Risk-based",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = AttributeSuppressionOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy1_field"],
    
    # Risk-based configuration
    ka_risk_field='k_anonymity',
    risk_threshold=5,
    
    # Additional fields to suppress
    additional_fields=['first_name'],
    
    # Suppression mode
    suppression_mode='REMOVE',
    save_suppressed_schema=True,
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_risk_based',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_risk_based' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    
    # Count columns suppressed
    cols_removed = len(df.columns) - len(result_df_s1.columns)
    print(f"\n📊 Columns removed: {cols_removed} (from {len(df.columns)} to {len(result_df_s1.columns)})")

## STRATEGY 2: Conditional Suppression Based on Field

**How to use:**
- Suppresses phone numbers for patients under 10
- Run to execute conditional strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → evaluate condition → suppress → metrics → visualize → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `condition_field='years_of_experience'`: Field to evaluate
- `condition_values=[10]`: Threshold value
- `condition_operator='lt'`: Less than comparison
- `additional_fields=['email']`: Also suppress email for minors

**Note:** Best for regulatory compliance (e.g., protecting minors' PII)

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: CONDITIONAL SUPPRESSION BASED ON AGE")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Conditional",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = AttributeSuppressionOperation(
    field_name=FIELD_CONFIG["strategy2_field"],
    
    # Conditional configuration
    condition_field='years_of_experience',
    condition_values=[18],
    condition_operator='lt',
    
    # Additional fields
    additional_fields=['email'],
    
    # Suppression mode
    suppression_mode='REMOVE',
    save_suppressed_schema=True,
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_conditional',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results (NEWEST file)
output_files_s2 = sorted(
    list((task_dir / 'strategy2_conditional' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    
    # Count conditional matches
    if 'years_of_experience' in result_df_s2.columns:
        minors = (result_df_s2['years_of_experience'] < 10).sum()
        print(f"\n📊 Protected {minors} minor records (years_of_experience < 10)")

## STRATEGY 3: Multi-Condition Suppression with AND/OR Logic

**How to use:**
- Suppresses email for high-risk OR critical diagnosis patients
- Run to execute multi-condition strategy
- Monitor progress bar (6 steps)

**What you'll see:**
- Configuration summary
- Progress: validation → evaluate conditions → combine logic → suppress → metrics → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `multi_conditions`: List of condition dictionaries
  - Condition 1: `risk_level == 'High'`
  - Condition 2: `diagnosis == 'Heart Disease'`
- `condition_logic='OR'`: Match any condition (vs 'AND' for all)
- `additional_fields=['insurance_id']`: Suppress related identifiers

**Note:** Optimal for complex privacy rules with multiple criteria

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: MULTI-CONDITION SUPPRESSION WITH OR LOGIC")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Multi-condition",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = AttributeSuppressionOperation(
    field_name=FIELD_CONFIG["strategy3_field"],
    
    # Multi-condition configuration
    multi_conditions=[
        {
            'field': 'remote_preference',
            'operator': 'eq',
            'values': ['Hybrid']
        },
        {
            'field': 'work_authorization',
            'operator': 'eq',
            'values': ['Citizen']
        }
    ],
    condition_logic='OR',  # Match ANY condition
    
    # Additional fields
    additional_fields=['resume_id'],
    
    # Suppression mode
    suppression_mode='REMOVE',
    save_suppressed_schema=True,
    
    # Output configuration
    output_format='csv',
    
    # Processing settings
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_multi_condition',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results (NEWEST file)
output_files_s3 = sorted(
    list((task_dir / 'strategy3_multi_condition' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    
    # Count multi-condition matches
    if 'remote_preference' in result_df_s3.columns and 'work_authorization' in result_df_s3.columns:
        hybrid = (result_df_s3['remote_preference'] == 'Hybrid').sum()
        citizen = (result_df_s3['work_authorization'] == 'Citizen').sum()
        print(f"\n📊 Protected {hybrid} Hybrid + {citizen} Citizen records")

## Step 4: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and effectiveness

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Column reduction comparison
- Strategy characteristics summary

**Note:** Fastest strategy isn't always best - consider privacy requirements

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1: {elapsed_s1:6.2f}s")
print(f"  Strategy 2: {elapsed_s2:6.2f}s")
print(f"  Strategy 3: {elapsed_s3:6.2f}s")
print(f"  Total:      {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print("\n📊 Strategy Characteristics:")
print(f"  Strategy 1: Risk-based (k < 5), 3 columns suppressed")
print(f"  Strategy 2: Conditional (age < 18), 2 columns suppressed")
print(f"  Strategy 3: Multi-condition (OR logic), 2 columns suppressed")

## Step 5: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: Performance, suppression stats, column reduction %
2. **Suppressed Schema**: Metadata per removed column
3. **Visualizations**: Before/after column count charts

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_risk_based', 'Strategy 1: Risk-Based'),
    ('strategy2_conditional', 'Strategy 2: Conditional'),
    ('strategy3_multi_condition', 'Strategy 3: Multi-Condition')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*_metrics_*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Display suppression metrics
                if 'columns_suppressed' in metrics:
                    print(f"   Columns suppressed: {metrics['columns_suppressed']}")
                if 'suppressed_column_names' in metrics:
                    print(f"   Suppressed columns: {', '.join(metrics['suppressed_column_names'])}")
                if 'data_width_reduction' in metrics:
                    print(f"   Data width reduction: {metrics['data_width_reduction']:.2f}%")
                
                # Display performance
                if 'duration_seconds' in metrics:
                    print(f"   Duration: {metrics['duration_seconds']:.2f}s")
                    
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Suppressed Schema (NEWEST)
    schema_files = sorted(
        list(metrics_dir.glob('*_suppressed_columns_schema_*.json')),
        key=lambda x: x.stat().st_mtime, reverse=True
    )
    if schema_files:
        print(f"\n📋 SUPPRESSED SCHEMA: {schema_files[0].name}")
        try:
            with open(schema_files[0], 'r') as f:
                schema = json.load(f)
            print(f"   Metadata for {len(schema)} column(s)")
            for col_name, col_info in list(schema.items())[:3]:
                if isinstance(col_info, dict):
                    dtype = col_info.get('dtype', 'unknown')
                    unique = col_info.get('unique_count', 0)
                    print(f"   • {col_name}: {dtype}, {unique} unique values")
        except Exception as e:
            print(f"   ⚠️  Error: {e}")
    
    # 3. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Calculate Privacy Protection Metrics

**How to use:**
- Run AFTER all strategies complete
- Calculates data utility retention

**What you'll see:**
- Original data: column count, identifier count
- Anonymized data: remaining columns per strategy
- Utility retention ratios (higher is better for analysis)

**Privacy targets:**
- ≥30% reduction: Strong privacy protection
- 15-30% reduction: Moderate protection
- ≥70% utility: High analytical value retained

**Note:** Requires result_df_s{N} from strategy execution cells

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY PROTECTION METRICS")
print("=" * 80 + "\n")

def calculate_utility_retention(original_df: pd.DataFrame, anonymized_df: pd.DataFrame) -> dict:
    """Calculate data utility retention metrics."""
    original_cols = len(original_df.columns)
    anonymized_cols = len(anonymized_df.columns)
    cols_removed = original_cols - anonymized_cols
    
    return {
        'original_columns': original_cols,
        'anonymized_columns': anonymized_cols,
        'columns_removed': cols_removed,
        'reduction_pct': (cols_removed / original_cols * 100) if original_cols > 0 else 0,
        'utility_retention': (anonymized_cols / original_cols * 100) if original_cols > 0 else 0
    }

# Check if strategies completed
try:
    # Strategy 1 metrics
    if 'result_df_s1' in locals() and result_df_s1 is not None:
        print("📊 STRATEGY 1: RISK-BASED")
        metrics_s1 = calculate_utility_retention(df, result_df_s1)
        print(f"   Original columns: {metrics_s1['original_columns']}")
        print(f"   Remaining columns: {metrics_s1['anonymized_columns']}")
        print(f"   Reduction: {metrics_s1['reduction_pct']:.1f}%")
        print(f"   Utility retention: {metrics_s1['utility_retention']:.1f}%")
    
    # Strategy 2 metrics
    if 'result_df_s2' in locals() and result_df_s2 is not None:
        print("\n📊 STRATEGY 2: CONDITIONAL")
        metrics_s2 = calculate_utility_retention(df, result_df_s2)
        print(f"   Original columns: {metrics_s2['original_columns']}")
        print(f"   Remaining columns: {metrics_s2['anonymized_columns']}")
        print(f"   Reduction: {metrics_s2['reduction_pct']:.1f}%")
        print(f"   Utility retention: {metrics_s2['utility_retention']:.1f}%")
    
    # Strategy 3 metrics
    if 'result_df_s3' in locals() and result_df_s3 is not None:
        print("\n📊 STRATEGY 3: MULTI-CONDITION")
        metrics_s3 = calculate_utility_retention(df, result_df_s3)
        print(f"   Original columns: {metrics_s3['original_columns']}")
        print(f"   Remaining columns: {metrics_s3['anonymized_columns']}")
        print(f"   Reduction: {metrics_s3['reduction_pct']:.1f}%")
        print(f"   Utility retention: {metrics_s3['utility_retention']:.1f}%")
    
    print("\n🎯 Higher reduction = better privacy, higher utility = better analysis capability")
        
except NameError:
    print("⚠️  Run Step 3 (Setup Environment) first!")

## Step 7: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports anonymized datasets for each strategy

**What you'll see (per strategy):**
- Available columns list
- Export path
- Preview (first 5 rows)
- Statistics (columns remaining, reduction %)

**Output structure:**
```
advanced_output/
├── strategy1_risk_based/
│   └── suppressed_data.csv
├── strategy2_conditional/
│   └── suppressed_data.csv
└── strategy3_multi_condition/
    └── suppressed_data.csv
```

**Note:** Files contain only non-suppressed columns for each strategy

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Base export directory
export_base_dir = task_dir
task_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Export directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: Risk-Based
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: RISK-BASED SUPPRESSION")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_risk_based'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    # Save
    output_path_s1 = s1_dir / 'suppressed_data.csv'
    try:
        result_df_s1.to_csv(output_path_s1, index=False)
        print(f"\n✅ Saved: {output_path_s1}")
        print(f"   Rows: {len(result_df_s1):,}")
        print(f"   Columns: {len(result_df_s1.columns)}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    # Preview
    print(f"\n📊 Preview:")
    print(result_df_s1.head())
    
    # Statistics
    cols_removed = len(df.columns) - len(result_df_s1.columns)
    print(f"\n📈 Columns removed: {cols_removed} ({cols_removed/len(df.columns)*100:.1f}%)")
else:
    print("\n⚠️  Strategy 1 data not available")

# ============================================================================
# STRATEGY 2: Conditional
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: CONDITIONAL SUPPRESSION")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_conditional'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    # Save
    output_path_s2 = s2_dir / 'suppressed_data.csv'
    try:
        result_df_s2.to_csv(output_path_s2, index=False)
        print(f"\n✅ Saved: {output_path_s2}")
        print(f"   Rows: {len(result_df_s2):,}")
        print(f"   Columns: {len(result_df_s2.columns)}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    # Preview
    print(f"\n📊 Preview:")
    print(result_df_s2.head())
    
    # Statistics
    cols_removed = len(df.columns) - len(result_df_s2.columns)
    print(f"\n📈 Columns removed: {cols_removed} ({cols_removed/len(df.columns)*100:.1f}%)")
else:
    print("\n⚠️  Strategy 2 data not available")

# ============================================================================
# STRATEGY 3: Multi-Condition
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: MULTI-CONDITION SUPPRESSION")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_multi_condition'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    # Save
    output_path_s3 = s3_dir / 'suppressed_data.csv'
    try:
        result_df_s3.to_csv(output_path_s3, index=False)
        print(f"\n✅ Saved: {output_path_s3}")
        print(f"   Rows: {len(result_df_s3):,}")
        print(f"   Columns: {len(result_df_s3.columns)}")
    except PermissionError:
        print(f"\n⚠️  Cannot save (file may be open)")
    
    # Preview
    print(f"\n📊 Preview:")
    print(result_df_s3.head())
    
    # Statistics
    cols_removed = len(df.columns) - len(result_df_s3.columns)
    print(f"\n📈 Columns removed: {cols_removed} ({cols_removed/len(df.columns)*100:.1f}%)")
else:
    print("\n⚠️  Strategy 3 data not available")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 Files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 suppression strategies implemented and compared
- ✅ Privacy protection metrics calculated
- ✅ Performance benchmarking completed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Risk-based suppression targets vulnerable records (k < 5) for maximum protection
- Conditional suppression enables regulatory compliance (e.g., protecting minors)
- Multi-condition logic provides flexible privacy rules with AND/OR operators
- Suppressed schema metadata preserves audit trail for compliance

**Strategy recommendations:**
- **Use Strategy 1** when: You have k-anonymity scores and want to protect high-risk individuals
- **Use Strategy 2** when: Regulatory requirements mandate field-specific protection (age, jurisdiction)
- **Use Strategy 3** when: Complex privacy policies require multiple criteria evaluation

**Next steps:**
- Test with your own sensitive datasets
- Tune risk thresholds for optimal protection
- Combine with other anonymization operations
- Deploy to production environment

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)